In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

from datasets import Dataset, Audio
from transformers import AutoProcessor

In [2]:
HOME = Path.home()
BASE_DIR = Path.cwd()

BASE = HOME / "data_zeroth"
BASE_OUT = HOME / "data_zeroth_hf"

TRAIN_CSV = BASE / "zeroth_short_samples.csv"
TEST_CSV = BASE / "zeroth_test_short_samples.csv"
REF_CSV = BASE / "refs_full.csv"
AUDIO_DIR = HOME / "data_zeroth"

MODEL_ID_TAG = "small"
MODEL_ID = "openai/whisper-small"  # tokenizer 기준

MIN_SEC = 1.0
MAX_SEC = 15.0

print("TRAIN_CSV:", TRAIN_CSV)
print("TEST_CSV:", TRAIN_CSV)
print("AUDIO_DIR:", AUDIO_DIR)

TRAIN_CSV: /home/data/bootcamp/user2/data_zeroth/zeroth_short_samples.csv
TEST_CSV: /home/data/bootcamp/user2/data_zeroth/zeroth_short_samples.csv
AUDIO_DIR: /home/data/bootcamp/user2/data_zeroth


In [3]:
import pandas as pd
import os
from pathlib import Path

df = pd.read_csv(TRAIN_CSV)
df = df[df["split"] == "train"].reset_index(drop=True)

# 컬럼명 통일
df = df.rename(columns={"text": "transcript"})

# audio_path: CSV에 들어있는 상대경로(예: 003/127/127_003_0010.wav)
df["audio_path"] = df["audio_path"].astype(str).str.strip()

# file: basename (정답 매칭용으로 쓰기 좋음)
df["file"] = df["audio_path"].apply(lambda p: os.path.basename(p))

# wav_path: 하위 폴더 포함한 실제 경로
# AUDIO_DIR은 예: Path("~/data_zeroth").expanduser() / "train_data_01" (또는 test_data_01)
df["wav_path"] = df["audio_path"].apply(lambda p: (AUDIO_DIR / p).resolve())

# wav 실제 존재하는 것만
df = df[df["wav_path"].apply(lambda p: Path(p).exists())]

print("usable rows:", len(df))
df.head()

usable rows: 22262


,split,audio_path,duration,transcript,file,wav_path
0,train,/home/data/bootcamp/data/zeroth/train_data_01/...,8.010,국내 토지보상제도의 근간은 공익사업을 위한 토지 등의 취득 및 보상에 관한 법률이다,127_003_0010.wav,/home/data/bootcamp/data/zeroth/train_data_01/...
1,train,/home/data/bootcamp/data/zeroth/train_data_01/...,6.080,갈탄은 대부분 발전소로 보내져서 에너지를 생산하는 데 사용됩니다,127_003_0018.wav,/home/data/bootcamp/data/zeroth/train_data_01/...
2,train,/home/data/bootcamp/data/zeroth/train_data_01/...,6.492,그밖의 자세한 사항은 중앙선거여론조사심의위원회 홈페이지를 참조하면 된다,127_003_0021.wav,/home/data/bootcamp/data/zeroth/train_data_01/...
3,train,/home/data/bootcamp/data/zeroth/train_data_01/...,5.696,교만해져서 이상한 짓을 저질러요 우리 역사적으로 보면,127_003_0104.wav,/home/data/bootcamp/data/zeroth/train_data_01/...
4,train,/home/data/bootcamp/data/zeroth/train_data_01/...,6.194,우선 이름 주소 전화번호 전자 우편 주소를 기재하십시오,127_003_0118.wav,/home/data/bootcamp/data/zeroth/train_data_01/...


In [4]:
from sklearn.model_selection import train_test_split

# 재현성 확보 (중요)
SEED = 42

df_train, df_eval = train_test_split(
    df,
    test_size=0.05,      # 5% → eval
    random_state=SEED,
    shuffle=True
)

print("train rows:", len(df_train))
print("eval rows :", len(df_eval))

df_train.head(), df_eval.head()

train rows: 21148
eval rows : 1114


(       split                                         audio_path  duration  \
 8020   train  /home/data/bootcamp/data/zeroth/train_data_01/...     5.157   
 14155  train  /home/data/bootcamp/data/zeroth/train_data_01/...     7.206   
 14720  train  /home/data/bootcamp/data/zeroth/train_data_01/...     7.570   
 12160  train  /home/data/bootcamp/data/zeroth/train_data_01/...    10.871   
 4641   train  /home/data/bootcamp/data/zeroth/train_data_01/...     6.771   
 
                                               transcript              file  \
 8020               지난해 낙찰가율 십 퍼센트 이하 상가들은 평균 십 일 차례 유찰됐다  185_003_2285.wav   
 14155     사실 그분은 자신을 제 병아리들을 날개 아래 모으는 어미 닭에 비하기까지 하셨습니다  152_003_0625.wav   
 14720   서울대에서 학점 인플레 현상이 유독 심한 원인으로는 느슨한 학점 시스템이 첫째로 꼽힌다  154_003_2709.wav   
 12160  한편 유상증자 발행주식 중 이십 퍼센트는 우리사주조합에 우선 배정되고 나머지 팔십 ...  201_003_2070.wav   
 4641          특히 이 위원장은 김대중 정신은 누구도 사유화 할 수 없고 독점해선 안 된다  159_003_1989.wav   
 
                                                 wav_p

In [5]:
import pandas as pd
import os
from pathlib import Path

df_test = pd.read_csv(TEST_CSV)

# 1) test 데이터만 사용 (안전하게)
df_test["split"] = df_test["split"].astype(str).str.strip().str.lower()
df_test = df_test[df_test["split"] == "test"].reset_index(drop=True)

# 2) file: basename (평가/merge 용)
df_test["file"] = df_test["wav_path"].apply(os.path.basename)

# 3) wav_path: 실제 파일 경로로 변환
# AUDIO_DIR 예: Path("~/data_zeroth/test_data_01").expanduser()
df_test["wav_path"] = df_test["wav_path"].apply(
    lambda p: (AUDIO_DIR / p).resolve()
)

# 4) 실제 존재하는 wav만 사용
df_test = df_test[df_test["wav_path"].apply(lambda p: p.exists())]

print("usable rows (test only):", len(df_test))
df_test.head()

usable rows (test only): 456


,split,wav_path,duration_sec,transcript,file
0,test,/home/data/bootcamp/data/zeroth/test_data_01/0...,10.929,지난해 삼 월 김 전 장관의 동료인 장동련 홍익대 교수가 민간 자문단장으로 위촉되면...,104_003_0019.wav
1,test,/home/data/bootcamp/data/zeroth/test_data_01/0...,8.817,그 바람에 나의 몸도 겹쳐 쓰러지며 한창 피어난 노란 동백꽃 속으로 폭 파묻혀 버렸다,104_003_0193.wav
2,test,/home/data/bootcamp/data/zeroth/test_data_01/0...,10.432,현재 백화점과 영화관 등은 오픈해 영업하고 있고 테마파크 및 아파트 등의 공사는 이...,104_003_0253.wav
3,test,/home/data/bootcamp/data/zeroth/test_data_01/0...,11.073,또한 박 전 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용에 대한 입장뿐 아니라...,104_003_0280.wav
4,test,/home/data/bootcamp/data/zeroth/test_data_01/0...,6.865,한 잔의 커피가 소비자에게 전해지기 위해선 여러 차례의 공정을 거쳐야 한다,104_003_0294.wav


In [6]:
print("train:", len(df_train))
print("eval :", len(df_eval))

train: 21148
eval : 1114


In [7]:
from datasets import Dataset

def to_hf_dataset(df: pd.DataFrame):
    return Dataset.from_dict({
        "audio_path": [str(p) for p in df["wav_path"]],
        "text": df["transcript"].astype(str).tolist()
    })

ds_train = to_hf_dataset(df_train)
ds_eval  = to_hf_dataset(df_eval)

ds_train, ds_eval

(Dataset({
     features: ['audio_path', 'text'],
     num_rows: 21148
 }),
 Dataset({
     features: ['audio_path', 'text'],
     num_rows: 1114
 }))

In [8]:
import soundfile as sf
import numpy as np

processor = AutoProcessor.from_pretrained(MODEL_ID)

def prepare_batch(batch):
    wav_path = batch["audio_path"]
    audio_array, sr = sf.read(wav_path)

    # stereo면 mono로
    if isinstance(audio_array, np.ndarray) and audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=1)

    # Whisper는 보통 16k 기준 (processor가 resample 처리 가능하지만, 여기서는 안전하게 그대로 넣고 sr 전달)
    batch["input_features"] = processor(
        audio_array,
        sampling_rate=sr
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

In [9]:
ds_train_prep = ds_train.map(
    prepare_batch,
    remove_columns=ds_train.column_names,
    num_proc=8
)

ds_eval_prep = ds_eval.map(
    prepare_batch,
    remove_columns=ds_eval.column_names,
    num_proc=8
)

Map (num_proc=8):   0%|          | 0/21148 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/1114 [00:00<?, ? examples/s]

In [10]:
OUT_DS = BASE_OUT / "hf_dataset_zeroth_whisper_small"

ds_train_prep.save_to_disk(OUT_DS / "train")
ds_eval_prep.save_to_disk(OUT_DS / "eval")

print("Saved to:", OUT_DS)

Saving the dataset (0/41 shards):   0%|          | 0/21148 [00:00<?, ? examples/s]

Saving the dataset (0/3 shards):   0%|          | 0/1114 [00:00<?, ? examples/s]

Saved to: /home/data/bootcamp/user2/data_zeroth_hf/hf_dataset_zeroth_whisper_small
